In [16]:

# ================================================================
# NUTRIAI
# AI-POWERED PERSONAL NUTRITION & DIET PLANNER
#
# GOOGLE COLAB — COMPLETE SINGLE-CELL APPLICATION
#
# Features:
# ✓ Professional dashboard
# ✓ SQLite persistent database
# ✓ User registration/login
# ✓ Personal profile
# ✓ BMR/TDEE calculation
# ✓ Calorie & macro targets
# ✓ Personalized diet-plan generator
# ✓ Food intake tracking
# ✓ REAL 7-day analytics
# ✓ Macro distribution chart
# ✓ Meal distribution chart
# ✓ Actual-data-only dashboard
# ✓ CSV export
# ✓ Gradio 6 compatible
#
# IMPORTANT:
# 1. Restart the Colab runtime before running this cell if you
#    previously installed incompatible NumPy/Pandas versions.
# 2. Run this entire cell from beginning to end.
# ================================================================


# ================================================================
# 1. INSTALL COLAB-SAFE DEPENDENCIES
# ================================================================

import subprocess
import sys


def pip_install(package, no_deps=False, force_reinstall=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if no_deps:
        cmd.append("--no-deps")
    if force_reinstall:
        cmd.append("--force-reinstall")
    cmd.append(package)
    subprocess.check_call(cmd)


# Colab-safe dependency handling.  Do not use pip's resolver to
# upgrade the complete scientific stack.  If a previous run left
# Pandas/NumPy at incompatible versions, repair only those packages.
from importlib.metadata import version as _pkg_version, PackageNotFoundError

def _version_tuple(name):
    try:
        return tuple(int(x) for x in _pkg_version(name).split(".")[:2])
    except (PackageNotFoundError, ValueError):
        return (0, 0)

_pd_parts = _version_tuple("pandas")
_np_parts = _version_tuple("numpy")

if _pd_parts != (2, 2):
    pip_install("pandas==2.2.3", no_deps=True)

if not (1 <= _np_parts[0] < 2 or (_np_parts[0] == 2 and _np_parts[1] < 3)):
    pip_install("numpy==2.2.6", no_deps=True)

# Install only the UI/plotting packages. NumPy/Pandas are pinned separately
# above with --no-deps so pip does not replace Colab's dependency stack.
pip_install("gradio>=6.0,<7.0")
# Plotly is intentionally not used. Matplotlib is used for charts to avoid
# Plotly validator conflicts in Colab runtimes.


# ================================================================
# 2. IMPORTS
# ================================================================

import os
import json
import hashlib
import sqlite3
import time

from pathlib import Path
from datetime import datetime, date, timedelta

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import gradio as gr


# ================================================================
# 3. ENVIRONMENT INFORMATION
# ================================================================

print("=" * 70)
print("NUTRIAI — ENVIRONMENT")
print("=" * 70)

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Gradio :", gr.__version__)

print("=" * 70)


# ================================================================
# 4. APPLICATION DIRECTORIES
# ================================================================

APP_DIR = Path("/content/NutriAI")

DATABASE_DIR = APP_DIR / "database"
EXPORT_DIR = APP_DIR / "exports"

DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATABASE_PATH = DATABASE_DIR / "nutriai.db"


# ================================================================
# 5. DATABASE
# ================================================================

def get_connection():

    conn = sqlite3.connect(
        DATABASE_PATH,
        check_same_thread=False
    )

    conn.row_factory = sqlite3.Row

    return conn


DB = get_connection()


# ================================================================
# 6. DATABASE TABLES
# ================================================================

DB.executescript("""

CREATE TABLE IF NOT EXISTS users (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    username TEXT UNIQUE NOT NULL,

    password_hash TEXT NOT NULL,

    created_at TEXT NOT NULL

);


CREATE TABLE IF NOT EXISTS profiles (

    username TEXT PRIMARY KEY,

    name TEXT,

    age INTEGER,

    sex TEXT,

    height REAL,

    weight REAL,

    activity TEXT,

    diet TEXT,

    goal TEXT,

    allergies TEXT,

    cuisine TEXT,

    budget REAL,

    updated_at TEXT

);


CREATE TABLE IF NOT EXISTS intake (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    username TEXT NOT NULL,

    log_date TEXT NOT NULL,

    food TEXT NOT NULL,

    grams REAL NOT NULL,

    calories REAL NOT NULL,

    protein REAL NOT NULL,

    carbs REAL NOT NULL,

    fat REAL NOT NULL,

    created_at TEXT NOT NULL

);


CREATE TABLE IF NOT EXISTS plans (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    username TEXT NOT NULL,

    created_at TEXT NOT NULL,

    goal TEXT,

    calories REAL,

    protein REAL,

    carbs REAL,

    fat REAL,

    plan_json TEXT

);

""")


DB.commit()


# ================================================================
# 7. SESSION
# ================================================================

SESSION = {
    "username": None
}


# ================================================================
# 8. FOOD DATABASE
# ================================================================

FOODS = [

    ["Oats", 389, 16.9, 66.3, 6.9,
     "Breakfast",
     "vegetarian,vegan,non-vegetarian",
     "gluten"],

    ["Banana", 89, 1.1, 22.8, 0.3,
     "Snack",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Apple", 52, 0.3, 14.0, 0.2,
     "Snack",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Orange", 47, 0.9, 11.8, 0.1,
     "Snack",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Cucumber", 15, 0.7, 3.6, 0.1,
     "Snack",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Egg", 155, 13.0, 1.1, 11.0,
     "Breakfast",
     "non-vegetarian",
     ""],

    ["Milk", 61, 3.2, 4.8, 3.3,
     "Breakfast",
     "vegetarian,non-vegetarian",
     "lactose"],

    ["Greek Yogurt", 59, 10.0, 3.6, 0.4,
     "Snack",
     "vegetarian,non-vegetarian",
     "lactose"],

    ["Almonds", 579, 21.2, 21.6, 49.9,
     "Snack",
     "vegetarian,vegan,non-vegetarian",
     "nuts"],

    ["Peanut Butter", 588, 25.0, 20.0, 50.0,
     "Snack",
     "vegetarian,vegan,non-vegetarian",
     "nuts"],

    ["Rice", 130, 2.7, 28.2, 0.3,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Brown Rice", 123, 2.7, 25.6, 1.0,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Roti", 297, 11.0, 46.0, 7.5,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     "gluten"],

    ["Dal", 116, 9.0, 20.0, 0.4,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Moong Dal", 105, 7.0, 19.0, 0.4,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Rajma", 127, 8.7, 22.8, 0.5,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Chickpeas", 164, 8.9, 27.4, 2.6,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Paneer", 265, 18.0, 6.0, 20.0,
     "Lunch",
     "vegetarian,non-vegetarian",
     "lactose"],

    ["Tofu", 76, 8.0, 1.9, 4.8,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     "soy"],

    ["Chicken Breast", 165, 31.0, 0.0, 3.6,
     "Lunch",
     "non-vegetarian",
     ""],

    ["Chicken Curry", 180, 25.0, 4.0, 7.0,
     "Dinner",
     "non-vegetarian",
     ""],

    ["Fish", 128, 26.0, 0.0, 2.7,
     "Dinner",
     "non-vegetarian",
     ""],

    ["Fish Curry", 150, 20.0, 5.0, 5.0,
     "Dinner",
     "non-vegetarian",
     ""],

    ["Quinoa", 120, 4.4, 21.3, 1.9,
     "Dinner",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Vegetable Soup", 45, 2.0, 8.0, 0.5,
     "Dinner",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Mixed Vegetables", 65, 3.0, 12.0, 1.0,
     "Dinner",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Salad", 35, 1.5, 7.0, 0.2,
     "Dinner",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Sweet Potato", 86, 1.6, 20.1, 0.1,
     "Lunch",
     "vegetarian,vegan,non-vegetarian",
     ""],

    ["Vegetable Khichdi", 120, 4.0, 20.0, 2.5,
     "Dinner",
     "vegetarian,vegan,non-vegetarian",
     ""],

]


FOOD_DF = pd.DataFrame(
    FOODS,
    columns=[
        "food",
        "calories",
        "protein",
        "carbs",
        "fat",
        "meal",
        "diets",
        "allergens"
    ]
)

FOOD_NAMES = FOOD_DF["food"].tolist()


# ================================================================
# 9. PASSWORD HASH
# ================================================================

def hash_password(password):

    return hashlib.sha256(
        str(password).encode("utf-8")
    ).hexdigest()


# ================================================================
# 10. REGISTER
# ================================================================

def register_user(username, password):

    username = (username or "").strip()
    password = password or ""

    if len(username) < 3:

        return "⚠️ Username must contain at least 3 characters."

    if len(password) < 4:

        return "⚠️ Password must contain at least 4 characters."

    try:

        DB.execute(
            """
            INSERT INTO users
            (
                username,
                password_hash,
                created_at
            )
            VALUES (?,?,?)
            """,
            (
                username,
                hash_password(password),
                datetime.now().isoformat()
            )
        )

        DB.commit()

        return """
### ✅ Account Created

Your NutriAI account has been created.

You can now sign in.
"""

    except sqlite3.IntegrityError:

        return """
### ⚠️ Username Already Exists

Please choose another username.
"""


# ================================================================
# 11. LOGIN
# ================================================================

def login_user(username, password):

    username = (username or "").strip()
    password = password or ""

    row = DB.execute(
        """
        SELECT *
        FROM users
        WHERE username=?
        AND password_hash=?
        """,
        (
            username,
            hash_password(password)
        )
    ).fetchone()

    if row:

        SESSION["username"] = username

        return f"""
### 🟢 Signed In

Welcome back, **{username}**.

Open the **Dashboard** tab to view your nutrition data.
"""

    return """
### ❌ Login Failed

Incorrect username or password.
"""


# ================================================================
# 12. LOGOUT
# ================================================================

def logout_user():

    SESSION["username"] = None

    return """
### 👋 Signed Out

Your session has been closed.
"""


# ================================================================
# 13. ACTIVITY FACTORS
# ================================================================

ACTIVITY_FACTORS = {

    "Sedentary": 1.20,

    "Lightly Active": 1.375,

    "Moderately Active": 1.55,

    "Very Active": 1.725

}


# ================================================================
# 14. CALORIE / MACRO ENGINE
# ================================================================

def calculate_targets(
    age,
    sex,
    height,
    weight,
    activity,
    goal
):

    age = float(age)
    height = float(height)
    weight = float(weight)

    # Mifflin-St Jeor equation

    if sex == "Male":

        bmr = (
            10 * weight
            + 6.25 * height
            - 5 * age
            + 5
        )

    else:

        bmr = (
            10 * weight
            + 6.25 * height
            - 5 * age
            - 161
        )

    factor = ACTIVITY_FACTORS.get(
        activity,
        1.55
    )

    tdee = bmr * factor

    if goal == "Weight Loss":

        calories = tdee - 350

    elif goal == "Weight Gain":

        calories = tdee + 300

    elif goal == "Muscle & Fitness":

        calories = tdee + 150

    else:

        calories = tdee

    calories = max(
        1200,
        round(calories)
    )

    # 30 / 40 / 30 macro split

    protein = round(
        calories * 0.30 / 4
    )

    carbs = round(
        calories * 0.40 / 4
    )

    fat = round(
        calories * 0.30 / 9
    )

    return {

        "bmr": round(bmr),

        "tdee": round(tdee),

        "calories": calories,

        "protein": protein,

        "carbs": carbs,

        "fat": fat

    }


# ================================================================
# 15. PROFILE SAVE
# ================================================================

def save_profile(
    name,
    age,
    sex,
    height,
    weight,
    activity,
    diet,
    goal,
    allergies,
    cuisine,
    budget
):

    username = SESSION["username"]

    if not username:

        return """
### 🔒 Login Required

Please sign in before saving your profile.
"""

    try:

        age = int(age)
        height = float(height)
        weight = float(weight)
        budget = float(budget or 0)

    except Exception:

        return """
### ⚠️ Invalid Data

Please enter valid numeric values.
"""

    DB.execute(
        """
        INSERT INTO profiles
        (
            username,
            name,
            age,
            sex,
            height,
            weight,
            activity,
            diet,
            goal,
            allergies,
            cuisine,
            budget,
            updated_at
        )
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)

        ON CONFLICT(username)
        DO UPDATE SET

            name=excluded.name,
            age=excluded.age,
            sex=excluded.sex,
            height=excluded.height,
            weight=excluded.weight,
            activity=excluded.activity,
            diet=excluded.diet,
            goal=excluded.goal,
            allergies=excluded.allergies,
            cuisine=excluded.cuisine,
            budget=excluded.budget,
            updated_at=excluded.updated_at
        """,
        (
            username,
            name,
            age,
            sex,
            height,
            weight,
            activity,
            diet,
            goal,
            allergies,
            cuisine,
            budget,
            datetime.now().isoformat()
        )
    )

    DB.commit()

    return """
### ✅ Profile Updated

Your nutrition targets will use the latest profile information.
"""


# ================================================================
# 16. GET PROFILE
# ================================================================

def get_profile():

    username = SESSION["username"]

    if not username:

        return None

    return DB.execute(
        """
        SELECT *
        FROM profiles
        WHERE username=?
        """,
        (username,)
    ).fetchone()


# ================================================================
# 17. FILTER FOOD DATABASE
# ================================================================

def get_available_foods(
    diet,
    allergies
):

    allergy_list = [

        x.strip().lower()

        for x in
        (allergies or "").split(",")

        if x.strip()

    ]

    selected = []

    for _, food in FOOD_DF.iterrows():

        if diet not in food["diets"]:

            continue

        allergens = [

            x.strip().lower()

            for x in
            food["allergens"].split(",")

            if x.strip()

        ]

        if any(
            a in allergens
            for a in allergy_list
        ):

            continue

        selected.append(food)

    if not selected:

        return FOOD_DF.copy()

    return pd.DataFrame(selected)


# ================================================================
# 18. GENERATE DIET PLAN
# ================================================================

def generate_plan_data(
    age,
    sex,
    height,
    weight,
    activity,
    diet,
    goal,
    allergies
):

    targets = calculate_targets(
        age,
        sex,
        height,
        weight,
        activity,
        goal
    )

    available = get_available_foods(
        diet,
        allergies
    )

    meal_percentages = {

        "Breakfast": 0.25,

        "Lunch": 0.30,

        "Snack": 0.15,

        "Dinner": 0.30

    }

    meals = {}

    for meal, percentage in meal_percentages.items():

        candidates = available[
            available["meal"] == meal
        ]

        if candidates.empty:

            candidates = available

        food = candidates.iloc[0]

        target_calories = (
            targets["calories"]
            * percentage
        )

        grams = (
            target_calories
            / max(
                food["calories"],
                1
            )
            * 100
        )

        grams = max(
            40,
            min(
                grams,
                450
            )
        )

        multiplier = grams / 100

        meals[meal] = {

            "food": food["food"],

            "grams": round(grams),

            "calories": round(
                food["calories"]
                * multiplier
            ),

            "protein": round(
                food["protein"]
                * multiplier,
                1
            ),

            "carbs": round(
                food["carbs"]
                * multiplier,
                1
            ),

            "fat": round(
                food["fat"]
                * multiplier,
                1
            )

        }

    total = {

        key: round(
            sum(
                meal[key]
                for meal in meals.values()
            ),
            1
        )

        for key in [
            "calories",
            "protein",
            "carbs",
            "fat"
        ]

    }

    return {

        "targets": targets,

        "meals": meals,

        "total": total

    }


# ================================================================
# 19. GENERATE / SAVE PLAN
# ================================================================

def generate_user_plan(
    name,
    age,
    sex,
    height,
    weight,
    activity,
    diet,
    goal,
    allergies,
    cuisine,
    budget
):

    username = SESSION["username"]

    if not username:

        return (
            "<h3>🔒 Please sign in first.</h3>",
            None
        )

    try:

        age = int(age)
        height = float(height)
        weight = float(weight)

    except Exception:

        return (
            "<h3>⚠️ Please enter valid profile data.</h3>",
            None
        )

    plan = generate_plan_data(
        age,
        sex,
        height,
        weight,
        activity,
        diet,
        goal,
        allergies
    )

    DB.execute(
        """
        INSERT INTO plans
        (
            username,
            created_at,
            goal,
            calories,
            protein,
            carbs,
            fat,
            plan_json
        )
        VALUES (?,?,?,?,?,?,?,?)
        """,
        (
            username,
            datetime.now().isoformat(),
            goal,
            plan["total"]["calories"],
            plan["total"]["protein"],
            plan["total"]["carbs"],
            plan["total"]["fat"],
            json.dumps(plan)
        )
    )

    DB.commit()

    return (
        render_plan(plan),
        json.dumps(plan)
    )


# ================================================================
# 20. PLAN HTML
# ================================================================

def render_plan(plan):

    target = plan["targets"]

    icons = {

        "Breakfast": "☀️",

        "Lunch": "🥗",

        "Snack": "🍎",

        "Dinner": "🌙"

    }

    html = f"""

<div class="plan-shell">

    <div class="plan-hero">

        <div>

            <div class="micro-label">
                PERSONALIZED NUTRITION ENGINE
            </div>

            <h1>
                Your Nutrition Blueprint
            </h1>

            <p>
                A meal structure generated from your
                profile, activity level and nutrition goal.
            </p>

        </div>

        <div class="plan-target">

            <span>DAILY TARGET</span>

            <strong>
                {target["calories"]}
            </strong>

            <small>
                kcal
            </small>

        </div>

    </div>


    <div class="target-grid">

        <div class="target-card">
            <span>🔥</span>
            <small>CALORIES</small>
            <strong>{target["calories"]}</strong>
            <em>kcal</em>
        </div>

        <div class="target-card">
            <span>🥩</span>
            <small>PROTEIN</small>
            <strong>{target["protein"]}</strong>
            <em>g</em>
        </div>

        <div class="target-card">
            <span>🍚</span>
            <small>CARBS</small>
            <strong>{target["carbs"]}</strong>
            <em>g</em>
        </div>

        <div class="target-card">
            <span>🥑</span>
            <small>FAT</small>
            <strong>{target["fat"]}</strong>
            <em>g</em>
        </div>

    </div>


    <div class="meal-grid">

"""

    for meal, item in plan["meals"].items():

        html += f"""

        <div class="meal-card">

            <div class="meal-icon">
                {icons[meal]}
            </div>

            <div class="meal-content">

                <div class="micro-label">
                    {meal.upper()}
                </div>

                <h3>
                    {item["food"]}
                </h3>

                <p>
                    {item["grams"]} g serving
                </p>

                <div class="meal-data">

                    <span>
                        <b>{item["calories"]}</b> kcal
                    </span>

                    <span>
                        P <b>{item["protein"]}g</b>
                    </span>

                    <span>
                        C <b>{item["carbs"]}g</b>
                    </span>

                    <span>
                        F <b>{item["fat"]}g</b>
                    </span>

                </div>

            </div>

        </div>

"""

    html += """

    </div>

    <div class="plan-note">

        <span>✦</span>

        <div>

            <strong>
                NutriAI Recommendation
            </strong>

            <p>
                Use this plan as a general wellness
                planning aid. Individual nutrition needs
                can vary substantially.
            </p>

        </div>

    </div>

</div>

"""

    return html


# ================================================================
# 21. ADD INTAKE
# ================================================================

def add_intake(
    food,
    grams,
    log_date
):

    username = SESSION["username"]

    if not username:

        return """
### 🔒 Login Required
"""

    try:

        grams = float(grams)

        if not log_date:

            log_date = str(date.today())

        # Validate date

        datetime.strptime(
            str(log_date),
            "%Y-%m-%d"
        )

    except Exception:

        return """
### ⚠️ Invalid quantity or date.

Use date format: YYYY-MM-DD
"""

    if grams <= 0:

        return """
### ⚠️ Quantity must be greater than zero.
"""

    row = FOOD_DF[
        FOOD_DF["food"] == food
    ]

    if row.empty:

        return """
### ⚠️ Food not found.
"""

    item = row.iloc[0]

    multiplier = grams / 100

    calories = (
        item["calories"]
        * multiplier
    )

    protein = (
        item["protein"]
        * multiplier
    )

    carbs = (
        item["carbs"]
        * multiplier
    )

    fat = (
        item["fat"]
        * multiplier
    )

    DB.execute(
        """
        INSERT INTO intake
        (
            username,
            log_date,
            food,
            grams,
            calories,
            protein,
            carbs,
            fat,
            created_at
        )
        VALUES (?,?,?,?,?,?,?,?,?)
        """,
        (
            username,
            str(log_date),
            food,
            grams,
            calories,
            protein,
            carbs,
            fat,
            datetime.now().isoformat()
        )
    )

    DB.commit()

    return f"""
### ✅ Intake Recorded

**{food} — {grams:.0f} g**

| Nutrition | Amount |
|---|---:|
| Calories | **{calories:.0f} kcal** |
| Protein | **{protein:.1f} g** |
| Carbohydrates | **{carbs:.1f} g** |
| Fat | **{fat:.1f} g** |

Date: **{log_date}**
"""


# ================================================================
# 22. TODAY INTAKE DATA
# ================================================================

def get_intake_for_date(log_date):

    username = SESSION["username"]

    if not username:

        return pd.DataFrame()

    rows = DB.execute(
        """
        SELECT
            log_date AS Date,
            food AS Food,
            ROUND(grams,1) AS Grams,
            ROUND(calories,1) AS Calories,
            ROUND(protein,1) AS Protein,
            ROUND(carbs,1) AS Carbs,
            ROUND(fat,1) AS Fat
        FROM intake
        WHERE username=?
        AND log_date=?
        ORDER BY id DESC
        """,
        (
            username,
            log_date
        )
    ).fetchall()

    return pd.DataFrame(
        [dict(row) for row in rows]
    )


# ================================================================
# 23. ALL INTAKE
# ================================================================

def get_all_intake():

    username = SESSION["username"]

    if not username:

        return pd.DataFrame()

    rows = DB.execute(
        """
        SELECT
            log_date AS Date,
            food AS Food,
            ROUND(grams,1) AS Grams,
            ROUND(calories,1) AS Calories,
            ROUND(protein,1) AS Protein,
            ROUND(carbs,1) AS Carbs,
            ROUND(fat,1) AS Fat
        FROM intake
        WHERE username=?
        ORDER BY log_date DESC, id DESC
        """,
        (username,)
    ).fetchall()

    return pd.DataFrame(
        [dict(row) for row in rows]
    )


# ================================================================
# 24. DAILY TOTAL
# ================================================================

def get_daily_total(
    username,
    log_date
):

    row = DB.execute(
        """
        SELECT

            COALESCE(SUM(calories),0)
            AS calories,

            COALESCE(SUM(protein),0)
            AS protein,

            COALESCE(SUM(carbs),0)
            AS carbs,

            COALESCE(SUM(fat),0)
            AS fat

        FROM intake

        WHERE username=?
        AND log_date=?
        """,
        (
            username,
            log_date
        )
    ).fetchone()

    return {

        "calories": float(
            row["calories"]
        ),

        "protein": float(
            row["protein"]
        ),

        "carbs": float(
            row["carbs"]
        ),

        "fat": float(
            row["fat"]
        )

    }


# ================================================================
# 25. ACTUAL 7-DAY DATA
# ================================================================

def get_seven_day_data(
    username
):

    end = date.today()

    start = (
        end
        - timedelta(days=6)
    )

    rows = DB.execute(
        """
        SELECT

            log_date,

            COALESCE(
                SUM(calories),
                0
            ) AS calories,

            COALESCE(
                SUM(protein),
                0
            ) AS protein,

            COALESCE(
                SUM(carbs),
                0
            ) AS carbs,

            COALESCE(
                SUM(fat),
                0
            ) AS fat

        FROM intake

        WHERE username=?

        AND log_date BETWEEN ?
        AND ?

        GROUP BY log_date

        ORDER BY log_date
        """,
        (
            username,
            str(start),
            str(end)
        )
    ).fetchall()

    actual = {
        row["log_date"]: {
            "calories": float(row["calories"]),
            "protein": float(row["protein"]),
            "carbs": float(row["carbs"]),
            "fat": float(row["fat"])
        }
        for row in rows
    }

    records = []

    for i in range(7):

        current = (
            start
            + timedelta(days=i)
        )

        key = str(current)

        data = actual.get(
            key,
            {
                "calories": 0,
                "protein": 0,
                "carbs": 0,
                "fat": 0
            }
        )

        records.append({

            "date": key,

            "day": current.strftime("%a"),

            "calories": data["calories"],

            "protein": data["protein"],

            "carbs": data["carbs"],

            "fat": data["fat"],

            "recorded": key in actual

        })

    return pd.DataFrame(records)


# ================================================================
# 26. PROFILE TARGETS
# ================================================================

def get_targets_for_user():

    profile = get_profile()

    if not profile:

        return None

    return calculate_targets(
        profile["age"],
        profile["sex"],
        profile["height"],
        profile["weight"],
        profile["activity"],
        profile["goal"]
    )


# ================================================================
# 27. DASHBOARD DATA
# ================================================================

def dashboard_data():

    username = SESSION["username"]

    if not username:

        return {

            "logged_in": False

        }

    profile = get_profile()

    if not profile:

        return {

            "logged_in": True,

            "profile": False

        }

    targets = get_targets_for_user()

    today = str(date.today())

    consumed = get_daily_total(
        username,
        today
    )

    seven = get_seven_day_data(
        username
    )

    return {

        "logged_in": True,

        "profile": True,

        "profile_row": profile,

        "targets": targets,

        "consumed": consumed,

        "seven_day": seven

    }


# ================================================================
# 28. DASHBOARD HTML
# ================================================================

def build_dashboard_html():

    data = dashboard_data()

    if not data["logged_in"]:

        return """

<div class="empty-dashboard">

    <div class="empty-icon">
        🔐
    </div>

    <h2>
        Your Nutrition Command Center
    </h2>

    <p>
        Sign in to access your personalized
        nutrition analytics.
    </p>

</div>

"""

    if not data["profile"]:

        return """

<div class="empty-dashboard">

    <div class="empty-icon">
        👤
    </div>

    <h2>
        Complete Your Profile
    </h2>

    <p>
        Save your age, height, weight, activity,
        diet and goal in the Profile tab.
    </p>

</div>

"""

    profile = data["profile_row"]

    targets = data["targets"]

    consumed = data["consumed"]

    seven = data["seven_day"]

    name = (
        profile["name"]
        or data["profile_row"]["username"]
    )

    calories_pct = min(
        100,
        round(
            consumed["calories"]
            / max(
                targets["calories"],
                1
            )
            * 100
        )
    )

    protein_pct = min(
        100,
        round(
            consumed["protein"]
            / max(
                targets["protein"],
                1
            )
            * 100
        )
    )

    carbs_pct = min(
        100,
        round(
            consumed["carbs"]
            / max(
                targets["carbs"],
                1
            )
            * 100
        )
    )

    fat_pct = min(
        100,
        round(
            consumed["fat"]
            / max(
                targets["fat"],
                1
            )
            * 100
        )
    )

    remaining = max(
        0,
        round(
            targets["calories"]
            - consumed["calories"]
        )
    )

    today_has_data = (
        consumed["calories"] > 0
    )

    # Dynamic macro shares for the visual donut.
    macro_grams_total = (
        consumed["protein"] +
        consumed["carbs"] +
        consumed["fat"]
    )
    if macro_grams_total > 0:
        protein_share = consumed["protein"] / macro_grams_total * 100
        carbs_share = consumed["carbs"] / macro_grams_total * 100
    else:
        protein_share = 0
        carbs_share = 0
    carbs_end = protein_share + carbs_share

    status_text = (
        "On track"
        if calories_pct <= 100
        else
        "Above target"
    )

    return f"""

<div class="dash-shell">

    <!-- TOP BAR -->

    <div class="dash-top">

        <div>

            <div class="micro-label">
                NUTRIAI / OVERVIEW
            </div>

            <h1>
                Good day, {name} <span>✦</span>
            </h1>

            <p>
                Your nutrition command center for
                {date.today().strftime("%A, %d %B %Y")}.
            </p>

        </div>

        <div class="live-status">

            <span class="live-dot"></span>

            LIVE DATA

        </div>

    </div>


    <!-- KPI ROW -->

    <div class="metric-grid">

        <div class="metric-card metric-primary">

            <div class="metric-top">

                <span>
                    DAILY ENERGY
                </span>

                <b>
                    🔥
                </b>

            </div>

            <div class="metric-number">

                {consumed["calories"]:.0f}

                <small>
                    kcal
                </small>

            </div>

            <div class="metric-target">

                Target
                <strong>
                    {targets["calories"]:,} kcal
                </strong>

            </div>

            <div class="mini-progress">

                <i
                    style="width:{calories_pct}%"
                ></i>

            </div>

            <div class="metric-footer">

                <span>
                    {calories_pct}% consumed
                </span>

                <span>
                    {remaining:,} kcal left
                </span>

            </div>

        </div>


        <div class="metric-card">

            <div class="metric-top">

                <span>
                    PROTEIN
                </span>

                <b>
                    ◈
                </b>

            </div>

            <div class="metric-number">

                {consumed["protein"]:.1f}

                <small>
                    g
                </small>

            </div>

            <div class="metric-target">

                Target
                <strong>
                    {targets["protein"]} g
                </strong>

            </div>

            <div class="mini-progress protein-bar">

                <i
                    style="width:{protein_pct}%"
                ></i>

            </div>

            <div class="metric-footer">

                <span>
                    {protein_pct}% complete
                </span>

            </div>

        </div>


        <div class="metric-card">

            <div class="metric-top">

                <span>
                    CARBOHYDRATES
                </span>

                <b>
                    ◉
                </b>

            </div>

            <div class="metric-number">

                {consumed["carbs"]:.1f}

                <small>
                    g
                </small>

            </div>

            <div class="metric-target">

                Target
                <strong>
                    {targets["carbs"]} g
                </strong>

            </div>

            <div class="mini-progress carbs-bar">

                <i
                    style="width:{carbs_pct}%"
                ></i>

            </div>

            <div class="metric-footer">

                <span>
                    {carbs_pct}% complete
                </span>

            </div>

        </div>


        <div class="metric-card">

            <div class="metric-top">

                <span>
                    FAT
                </span>

                <b>
                    ◌
                </b>

            </div>

            <div class="metric-number">

                {consumed["fat"]:.1f}

                <small>
                    g
                </small>

            </div>

            <div class="metric-target">

                Target
                <strong>
                    {targets["fat"]} g
                </strong>

            </div>

            <div class="mini-progress fat-bar">

                <i
                    style="width:{fat_pct}%"
                ></i>

            </div>

            <div class="metric-footer">

                <span>
                    {fat_pct}% complete
                </span>

            </div>

        </div>

    </div>


    <!-- MAIN ANALYTICS -->

    <div class="analytics-grid">


        <div class="analytics-card energy-card">

            <div class="card-heading">

                <div>

                    <div class="micro-label">
                        ENERGY TRACKING
                    </div>

                    <h2>
                        Today's Calorie Progress
                    </h2>

                </div>

                <div class="status-pill">
                    {status_text}
                </div>

            </div>


            <div class="energy-body">

                <div class="energy-ring"
                    style="--p:{calories_pct * 3.6}deg">

                    <div>

                        <strong>
                            {calories_pct}%
                        </strong>

                        <span>
                            of target
                        </span>

                    </div>

                </div>


                <div class="energy-copy">

                    <div class="energy-value">

                        {consumed["calories"]:.0f}

                        <small>
                            / {targets["calories"]:,} kcal
                        </small>

                    </div>

                    <p>

                        {
                            "Start logging meals to see your daily progress."
                            if not today_has_data
                            else
                            f"{remaining:,} kcal remain within today's target."
                        }

                    </p>

                    <div class="energy-stats">

                        <div>
                            <span>Goal</span>
                            <b>{targets["calories"]:,}</b>
                        </div>

                        <div>
                            <span>Consumed</span>
                            <b>{consumed["calories"]:.0f}</b>
                        </div>

                        <div>
                            <span>Remaining</span>
                            <b>{remaining:,}</b>
                        </div>

                    </div>

                </div>

            </div>

        </div>


        <div class="analytics-card macro-card">

            <div class="card-heading">

                <div>

                    <div class="micro-label">
                        MACRO DISTRIBUTION
                    </div>

                    <h2>
                        Nutrition Balance
                    </h2>

                </div>

            </div>


            <div class="macro-layout">

                <div class="macro-donut" style="--protein-share:{protein_share:.2f}%; --carbs-end:{carbs_end:.2f}% ;">

                    <div class="donut-center">

                        <strong>
                            {consumed["protein"] + consumed["carbs"] + consumed["fat"]:.0f}
                        </strong>

                        <span>
                            grams
                        </span>

                    </div>

                </div>


                <div class="macro-legend">

                    <div>

                        <i class="dot protein-dot"></i>

                        <span>
                            Protein
                        </span>

                        <b>
                            {consumed["protein"]:.1f}g
                        </b>

                    </div>

                    <div>

                        <i class="dot carbs-dot"></i>

                        <span>
                            Carbs
                        </span>

                        <b>
                            {consumed["carbs"]:.1f}g
                        </b>

                    </div>

                    <div>

                        <i class="dot fat-dot"></i>

                        <span>
                            Fat
                        </span>

                        <b>
                            {consumed["fat"]:.1f}g
                        </b>

                    </div>

                </div>

            </div>

        </div>

    </div>


    <!-- ACTIVITY -->

    <div class="analytics-card trend-card">

        <div class="card-heading">

            <div>

                <div class="micro-label">
                    ACTUAL INTAKE HISTORY
                </div>

                <h2>
                    7-Day Calorie Trend
                </h2>

            </div>

            <div class="data-note">

                ● Database data only

            </div>

        </div>

        <div class="trend-info">

            <div>

                <strong>
                    {
                        seven["calories"].sum():.0f
                    }
                </strong>

                <span>
                    kcal logged in last 7 days
                </span>

            </div>

            <div>

                <strong>
                    {
                        int(
                            seven["recorded"].sum()
                        )
                    }
                    / 7
                </strong>

                <span>
                    days recorded
                </span>

            </div>

        </div>

    </div>


    <!-- MEAL SNAPSHOT -->

    <div class="analytics-card">

        <div class="card-heading">

            <div>

                <div class="micro-label">
                    DAILY JOURNEY
                </div>

                <h2>
                    Today's Intake
                </h2>

            </div>

        </div>

        <div class="meal-timeline">

            <div>
                <span>☀</span>
                <b>Breakfast</b>
            </div>

            <div class="line"></div>

            <div>
                <span>◈</span>
                <b>Lunch</b>
            </div>

            <div class="line"></div>

            <div>
                <span>○</span>
                <b>Snack</b>
            </div>

            <div class="line"></div>

            <div>
                <span>☾</span>
                <b>Dinner</b>
            </div>

        </div>

    </div>


    <!-- PROFILE SUMMARY -->

    <div class="profile-strip">

        <div>

            <span>
                GOAL
            </span>

            <strong>
                {profile["goal"]}
            </strong>

        </div>

        <div>

            <span>
                DIET
            </span>

            <strong>
                {profile["diet"].title()}
            </strong>

        </div>

        <div>

            <span>
                ACTIVITY
            </span>

            <strong>
                {profile["activity"]}
            </strong>

        </div>

        <div>

            <span>
                TDEE
            </span>

            <strong>
                {targets["tdee"]:,} kcal
            </strong>

        </div>

    </div>


    <div class="data-integrity">

        <span>
            ●
        </span>

        Dashboard values are calculated from your
        saved profile and recorded food intake.
        No synthetic/random intake data is displayed.

    </div>

</div>

"""


# ================================================================
# 29. DASHBOARD CHARTS
# ================================================================

# NOTE:
# The dashboard intentionally uses Matplotlib instead of Plotly.
# Some Google Colab runtimes can contain mixed Plotly installations
# after repeated pip installs, which can cause:
#
#   ImportError: cannot import name 'DataValidator'
#   from 'plotly.validators'
#
# Matplotlib is natively supported by Gradio's Plot component and
# avoids that dependency/validator problem completely.


def _dashboard_placeholder(title, message, height=360):
    """
    Create a professional empty-state chart using Matplotlib.
    This is returned when the user has not logged in, has no profile,
    or has not recorded enough data yet.
    """

    fig, ax = plt.subplots(figsize=(8.5, 4.0))

    fig.patch.set_alpha(0)
    ax.set_facecolor("none")

    ax.text(
        0.5,
        0.55,
        title,
        ha="center",
        va="center",
        fontsize=16,
        fontweight="bold",
        color="#334155",
        transform=ax.transAxes
    )

    ax.text(
        0.5,
        0.43,
        message,
        ha="center",
        va="center",
        fontsize=11,
        color="#64748B",
        transform=ax.transAxes,
        wrap=True
    )

    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout(pad=1.2)

    return fig


def build_dashboard_charts():

    data = dashboard_data()

    # ------------------------------------------------------------
    # SIGNED-OUT STATE
    # ------------------------------------------------------------

    if not data["logged_in"]:

        return (
            _dashboard_placeholder(
                "7-Day Calorie Trend",
                "Sign in to load your personal analytics."
            ),
            _dashboard_placeholder(
                "Macro Distribution",
                "Sign in and record intake to view your macro distribution.",
                height=340
            )
        )

    # ------------------------------------------------------------
    # PROFILE NOT COMPLETE
    # ------------------------------------------------------------

    if not data["profile"]:

        return (
            _dashboard_placeholder(
                "7-Day Calorie Trend",
                "Complete your nutrition profile first."
            ),
            _dashboard_placeholder(
                "Macro Distribution",
                "Complete your nutrition profile and record intake first.",
                height=340
            )
        )

    targets = data["targets"]
    seven = data["seven_day"]

    # ============================================================
    # 7-DAY CALORIE CHART
    # ============================================================

    fig_cal, ax_cal = plt.subplots(
        figsize=(8.5, 4.0)
    )

    fig_cal.patch.set_alpha(0)
    ax_cal.set_facecolor("none")

    days = seven["day"]
    calories = seven["calories"]

    # Actual intake
    ax_cal.plot(
        days,
        calories,
        marker="o",
        linewidth=3,
        markersize=7,
        color="#6C63FF",
        label="Actual intake"
    )

    # Light fill under actual intake
    try:
        ax_cal.fill_between(
            range(len(calories)),
            calories,
            0,
            color="#6C63FF",
            alpha=0.08
        )
    except Exception:
        pass

    # Daily target
    target_value = targets["calories"]

    ax_cal.axhline(
        target_value,
        color="#94A3B8",
        linewidth=2,
        linestyle="--",
        label="Daily target"
    )

    ax_cal.set_ylabel(
        "Calories (kcal)",
        fontsize=10,
        color="#475569"
    )

    ax_cal.set_xticks(
        range(len(days))
    )

    ax_cal.set_xticklabels(
        days,
        fontsize=9
    )

    ax_cal.tick_params(
        axis="both",
        colors="#64748B",
        labelsize=9
    )

    ax_cal.grid(
        axis="y",
        alpha=0.18,
        linewidth=0.8
    )

    ax_cal.grid(
        axis="x",
        visible=False
    )

    ax_cal.spines["top"].set_visible(False)
    ax_cal.spines["right"].set_visible(False)

    ax_cal.spines["left"].set_color("#E2E8F0")
    ax_cal.spines["bottom"].set_color("#E2E8F0")

    ax_cal.legend(
        loc="upper left",
        frameon=False,
        ncol=2,
        fontsize=9
    )

    fig_cal.tight_layout(
        pad=1.2
    )

    # ============================================================
    # MACRO DISTRIBUTION
    # ============================================================

    today = data["consumed"]

    protein = max(
        float(today.get("protein", 0) or 0),
        0
    )

    carbs = max(
        float(today.get("carbs", 0) or 0),
        0
    )

    fat = max(
        float(today.get("fat", 0) or 0),
        0
    )

    macro_values = [
        protein * 4,
        carbs * 4,
        fat * 9
    ]

    macro_names = [
        "Protein",
        "Carbohydrates",
        "Fat"
    ]

    macro_total = sum(macro_values)

    if macro_total <= 0:

        fig_macro = _dashboard_placeholder(
            "Macro Distribution",
            "Record food intake today to see your macro distribution.",
            height=340
        )

    else:

        fig_macro, ax_macro = plt.subplots(
            figsize=(7.5, 3.8)
        )

        fig_macro.patch.set_alpha(0)
        ax_macro.set_facecolor("none")

        # Donut chart
        ax_macro.pie(
            macro_values,
            labels=macro_names,
            autopct=lambda pct: (
                f"{pct:.0f}%"
                if pct > 0
                else ""
            ),
            startangle=90,
            counterclock=False,
            wedgeprops={
                "width": 0.32,
                "edgecolor": "white",
                "linewidth": 2
            },
            textprops={
                "fontsize": 9,
                "color": "#334155"
            }
        )

        ax_macro.text(
            0,
            0,
            f"{macro_total:.0f}\nkcal",
            ha="center",
            va="center",
            fontsize=14,
            fontweight="bold",
            color="#334155"
        )

        ax_macro.set_aspect("equal")

        fig_macro.tight_layout(
            pad=1.0
        )

    return (
        fig_cal,
        fig_macro
    )


# ================================================================
# 30. ACTUAL INTAKE TABLE
# ================================================================


# ================================================================

def refresh_today_table():

    return get_intake_for_date(
        str(date.today())
    )


# ================================================================
# 31. SAVED PLANS
# ================================================================

def get_saved_plans():

    username = SESSION["username"]

    if not username:

        return pd.DataFrame()

    rows = DB.execute(
        """
        SELECT

            id AS ID,

            substr(
                created_at,
                1,
                19
            ) AS Created,

            goal AS Goal,

            ROUND(
                calories
            ) AS Calories,

            ROUND(
                protein,
                1
            ) AS Protein_g,

            ROUND(
                carbs,
                1
            ) AS Carbs_g,

            ROUND(
                fat,
                1
            ) AS Fat_g

        FROM plans

        WHERE username=?

        ORDER BY id DESC
        """,
        (username,)
    ).fetchall()

    return pd.DataFrame(
        [dict(row) for row in rows]
    )


# ================================================================
# 32. EXPORT
# ================================================================

def export_latest_plan():

    username = SESSION["username"]

    if not username:

        return None

    row = DB.execute(
        """
        SELECT *

        FROM plans

        WHERE username=?

        ORDER BY id DESC

        LIMIT 1
        """,
        (username,)
    ).fetchone()

    if not row:

        return None

    plan = json.loads(
        row["plan_json"]
    )

    records = []

    for meal, item in plan["meals"].items():

        records.append({

            "Meal": meal,

            "Food": item["food"],

            "Quantity_g": item["grams"],

            "Calories_kcal": item["calories"],

            "Protein_g": item["protein"],

            "Carbohydrates_g": item["carbs"],

            "Fat_g": item["fat"]

        })

    filename = (

        EXPORT_DIR

        / (
            f"NutriAI_{username}_"
            f"{int(time.time())}.csv"
        )

    )

    pd.DataFrame(
        records
    ).to_csv(
        filename,
        index=False
    )

    return str(filename)


# ================================================================
# 33. PROFESSIONAL CSS
# ================================================================

CSS = r"""

/* ============================================================
   GLOBAL
   ============================================================ */

* {
    box-sizing: border-box;
}

body {
    background:
        #F4F6FA !important;

    font-family:
        Inter,
        ui-sans-serif,
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif !important;
}

.gradio-container {
    max-width:
        1500px !important;

    margin:
        auto !important;

    padding:
        20px !important;
}

footer {
    display:
        none !important;
}


/* ============================================================
   APP HEADER
   ============================================================ */

.app-header {

    position:
        relative;

    overflow:
        hidden;

    padding:
        42px;

    margin-bottom:
        20px;

    border-radius:
        28px;

    color:
        white;

    background:
        linear-gradient(
            135deg,
            #0E1330 0%,
            #252B68 55%,
            #635BFF 100%
        );

    box-shadow:
        0 25px 60px
        rgba(37,43,104,.20);
}

.app-header::before {

    content:
        "";

    position:
        absolute;

    width:
        260px;

    height:
        260px;

    right:
        -70px;

    top:
        -120px;

    border-radius:
        50%;

    background:
        rgba(255,255,255,.08);
}

.app-header::after {

    content:
        "";

    position:
        absolute;

    width:
        160px;

    height:
        160px;

    right:
        130px;

    bottom:
        -100px;

    border-radius:
        50%;

    background:
        rgba(255,255,255,.05);
}

.app-header h1 {
        color: #FFFFFF !important;

    color: #FFFFFF !important;

    position:
        relative;

    z-index:
        1;

    margin:
        0;

    font-size:
        42px;

    font-weight:
        850;

    letter-spacing:
        -1.5px;
}

.app-header p {

    position:
        relative;

    z-index:
        1;

    margin:
        8px 0 0;

    color:
        rgba(255,255,255,.72);

    font-size:
        14px;
}


/* ============================================================
   DASHBOARD
   ============================================================ */

.dash-shell {
    padding:
        5px 0 30px;
}

.dash-top {

    display:
        flex;

    justify-content:
        space-between;

    align-items:
        flex-start;

    margin:
        10px 0 24px;
}

.micro-label {

    color:
        #8A92A6;

    font-size:
        9px;

    font-weight:
        850;

    letter-spacing:
        1.7px;
}

.dash-top h1 {

    margin:
        5px 0;

    color:
        #111827;

    font-size:
        34px;

    font-weight:
        850;

    letter-spacing:
        -1px;
}

.dash-top h1 span {
    color:
        #6C63FF;
}

.dash-top p {

    margin:
        0;

    color:
        #7B8497;

    font-size:
        13px;
}

.live-status {

    display:
        flex;

    align-items:
        center;

    gap:
        8px;

    padding:
        9px 13px;

    border:
        1px solid #DDE2EA;

    border-radius:
        999px;

    background:
        white;

    color:
        #64748B;

    font-size:
        9px;

    font-weight:
        850;

    letter-spacing:
        1px;
}

.live-dot {

    width:
        7px;

    height:
        7px;

    border-radius:
        50%;

    background:
        #22C55E;

    box-shadow:
        0 0 0 4px
        rgba(34,197,94,.10);
}


/* ============================================================
   METRICS
   ============================================================ */

.metric-grid {

    display:
        grid;

    grid-template-columns:
        repeat(4,1fr);

    gap:
        14px;

    margin-bottom:
        16px;
}

.metric-card {

    padding:
        20px;

    min-height:
        185px;

    border:
        1px solid #E6E9F0;

    border-radius:
        20px;

    background:
        white;

    box-shadow:
        0 8px 28px
        rgba(15,23,42,.035);

    transition:
        transform .2s ease,
        box-shadow .2s ease;
}

.metric-card:hover {

    transform:
        translateY(-3px);

    box-shadow:
        0 15px 35px
        rgba(15,23,42,.08);
}

.metric-primary {

    color:
        white;

    background:
        linear-gradient(
            145deg,
            #171C43,
            #373C89
        );

    border:
        none;

    box-shadow:
        0 18px 40px
        rgba(37,43,104,.20);
}

.metric-top {

    display:
        flex;

    justify-content:
        space-between;

    align-items:
        center;

    color:
        #8A92A6;

    font-size:
        9px;

    font-weight:
        850;

    letter-spacing:
        1.3px;
}

.metric-primary .metric-top {
    color:
        rgba(255,255,255,.60);
}

.metric-top b {

    font-size:
        18px;

    color:
        #6C63FF;
}

.metric-primary .metric-top b {
    color:
        #A8A3FF;
}

.metric-number {

    margin-top:
        15px;

    color:
        #111827;

    font-size:
        33px;

    font-weight:
        850;

    letter-spacing:
        -1px;
}

.metric-primary .metric-number {
    color:
        white;
}

.metric-number small {

    font-size:
        12px;

    font-weight:
        600;

    color:
        #8A92A6;
}

.metric-primary .metric-number small {
    color:
        rgba(255,255,255,.60);
}

.metric-target {

    margin-top:
        4px;

    color:
        #9AA2B2;

    font-size:
        11px;
}

.metric-target strong {
    color:
        #4B5563;
}

.metric-primary .metric-target {
    color:
        rgba(255,255,255,.58);
}

.metric-primary .metric-target strong {
    color:
        rgba(255,255,255,.85);
}

.mini-progress {

    height:
        7px;

    margin-top:
        16px;

    overflow:
        hidden;

    border-radius:
        999px;

    background:
        #ECEEF3;
}

.metric-primary .mini-progress {
    background:
        rgba(255,255,255,.15);
}

.mini-progress i {

    display:
        block;

    height:
        100%;

    border-radius:
        inherit;

    background:
        #6C63FF;
}

.metric-primary .mini-progress i {
    background:
        #A8A3FF;
}

.protein-bar i {
    background:
        #7C70FF;
}

.carbs-bar i {
    background:
        #22C55E;
}

.fat-bar i {
    background:
        #F59E0B;
}

.metric-footer {

    display:
        flex;

    justify-content:
        space-between;

    margin-top:
        8px;

    color:
        #9AA2B2;

    font-size:
        9px;
}

.metric-primary .metric-footer {
    color:
        rgba(255,255,255,.58);
}


/* ============================================================
   ANALYTICS
   ============================================================ */

.analytics-grid {

    display:
        grid;

    grid-template-columns:
        1.35fr 1fr;

    gap:
        16px;

    margin-bottom:
        16px;
}

.analytics-card {

    padding:
        23px;

    margin-bottom:
        16px;

    border:
        1px solid #E6E9F0;

    border-radius:
        22px;

    background:
        white;

    box-shadow:
        0 8px 28px
        rgba(15,23,42,.035);
}

.card-heading {

    display:
        flex;

    align-items:
        flex-start;

    justify-content:
        space-between;
}

.card-heading h2 {

    margin:
        5px 0 0;

    color:
        #111827;

    font-size:
        19px;

    font-weight:
        800;
}

.status-pill {

    padding:
        7px 11px;

    border-radius:
        999px;

    background:
        #ECFDF3;

    color:
        #16804A;

    font-size:
        9px;

    font-weight:
        800;
}


/* ============================================================
   ENERGY
   ============================================================ */

.energy-body {

    display:
        flex;

    align-items:
        center;

    gap:
        35px;

    margin-top:
        28px;
}

.energy-ring {

    width:
        175px;

    height:
        175px;

    min-width:
        175px;

    display:
        flex;

    justify-content:
        center;

    align-items:
        center;

    border-radius:
        50%;

    background:
        conic-gradient(
            #6C63FF var(--p),
            #ECEEF5 0deg
        );
}

.energy-ring > div {

    width:
        133px;

    height:
        133px;

    display:
        flex;

    flex-direction:
        column;

    align-items:
        center;

    justify-content:
        center;

    border-radius:
        50%;

    background:
        white;
}

.energy-ring strong {

    color:
        #111827;

    font-size:
        34px;

    font-weight:
        850;
}

.energy-ring span {

    color:
        #98A0B1;

    font-size:
        10px;
}

.energy-value {

    color:
        #111827;

    font-size:
        32px;

    font-weight:
        850;
}

.energy-value small {

    color:
        #98A0B1;

    font-size:
        12px;

    font-weight:
        600;
}

.energy-copy p {

    max-width:
        320px;

    margin:
        8px 0 20px;

    color:
        #7B8497;

    font-size:
        12px;

    line-height:
        1.6;
}

.energy-stats {

    display:
        flex;

    gap:
        22px;
}

.energy-stats div {

    display:
        flex;

    flex-direction:
        column;

    gap:
        4px;
}

.energy-stats span {

    color:
        #9AA2B2;

    font-size:
        9px;
}

.energy-stats b {

    color:
        #374151;

    font-size:
        13px;
}


/* ============================================================
   MACRO
   ============================================================ */

.macro-layout {

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    gap:
        35px;

    margin-top:
        23px;
}

.macro-donut {

    width:
        175px;

    height:
        175px;

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    border-radius:
        50%;

    background:
        conic-gradient(
            #6C63FF 0 var(--protein-share, 0%),
            #22C55E var(--protein-share, 0%) var(--carbs-end, 0%),
            #F59E0B var(--carbs-end, 0%) 100%
        );
}

.donut-center {

    width:
        125px;

    height:
        125px;

    display:
        flex;

    flex-direction:
        column;

    align-items:
        center;

    justify-content:
        center;

    border-radius:
        50%;

    background:
        white;
}

.donut-center strong {

    color:
        #111827;

    font-size:
        25px;
}

.donut-center span {

    color:
        #98A0B1;

    font-size:
        9px;
}

.macro-legend {

    min-width:
        150px;
}

.macro-legend > div {

    display:
        grid;

    grid-template-columns:
        10px 1fr auto;

    align-items:
        center;

    gap:
        8px;

    padding:
        10px 0;

    border-bottom:
        1px solid #F0F1F5;
}

.macro-legend span {

    color:
        #687286;

    font-size:
        11px;
}

.macro-legend b {

    color:
        #111827;

    font-size:
        11px;
}

.dot {

    width:
        8px;

    height:
        8px;

    border-radius:
        50%;
}

.protein-dot {
    background:#6C63FF;
}

.carbs-dot {
    background:#22C55E;
}

.fat-dot {
    background:#F59E0B;
}


/* ============================================================
   TREND
   ============================================================ */

.trend-card {
    min-height:
        190px;
}

.data-note {

    color:
        #8A92A6;

    font-size:
        9px;

    padding:
        7px 10px;

    border-radius:
        8px;

    background:
        #F6F7FA;
}

.trend-info {

    display:
        flex;

    gap:
        60px;

    margin-top:
        25px;
}

.trend-info div {

    display:
        flex;

    flex-direction:
        column;

    gap:
        4px;
}

.trend-info strong {

    color:
        #111827;

    font-size:
        26px;
}

.trend-info span {

    color:
        #98A0B1;

    font-size:
        10px;
}


/* ============================================================
   MEAL TIMELINE
   ============================================================ */

.meal-timeline {

    display:
        flex;

    align-items:
        center;

    margin-top:
        25px;
}

.meal-timeline > div:not(.line) {

    display:
        flex;

    flex-direction:
        column;

    align-items:
        center;

    gap:
        7px;

    min-width:
        100px;
}

.meal-timeline span {

    width:
        40px;

    height:
        40px;

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    border-radius:
        12px;

    color:
        #6C63FF;

    background:
        #F0EFFF;

    font-size:
        16px;
}

.meal-timeline b {

    color:
        #687286;

    font-size:
        10px;
}

.meal-timeline .line {

    flex:
        1;

    height:
        1px;

    background:
        #E5E7EB;
}


/* ============================================================
   PROFILE STRIP
   ============================================================ */

.profile-strip {

    display:
        grid;

    grid-template-columns:
        repeat(4,1fr);

    gap:
        0;

    overflow:
        hidden;

    border:
        1px solid #E6E9F0;

    border-radius:
        18px;

    background:
        white;
}

.profile-strip > div {

    padding:
        18px 20px;

    border-right:
        1px solid #ECEEF2;
}

.profile-strip > div:last-child {
    border-right:
        none;
}

.profile-strip span {

    display:
        block;

    color:
        #9AA2B2;

    font-size:
        8px;

    font-weight:
        850;

    letter-spacing:
        1.2px;
}

.profile-strip strong {

    display:
        block;

    margin-top:
        6px;

    color:
        #374151;

    font-size:
        12px;
}


/* ============================================================
   DATA INTEGRITY
   ============================================================ */

.data-integrity {

    margin-top:
        14px;

    padding:
        13px 15px;

    border-radius:
        12px;

    color:
        #6B7280;

    background:
        #F8FAFC;

    font-size:
        9px;
}

.data-integrity span {

    color:
        #22C55E;

    margin-right:
        5px;
}


/* ============================================================
   EMPTY
   ============================================================ */

.empty-dashboard {

    padding:
        90px 25px;

    text-align:
        center;

    border:
        1px solid #E6E9F0;

    border-radius:
        25px;

    background:
        white;
}

.empty-icon {

    font-size:
        42px;

    margin-bottom:
        10px;
}

.empty-dashboard h2 {

    color:
        #111827;

    font-size:
        24px;
}

.empty-dashboard p {

    color:
        #7B8497;

    font-size:
        13px;
}


/* ============================================================
   PLAN
   ============================================================ */

.plan-shell {
    padding:
        4px 0 25px;
}

.plan-hero {

    display:
        flex;

    justify-content:
        space-between;

    align-items:
        center;

    padding:
        28px;

    margin-bottom:
        15px;

    border-radius:
        22px;

    background:
        linear-gradient(
            135deg,
            #121738,
            #30367C
        );

    color:
        white;
}

.plan-hero h1 {

    margin:
        6px 0;

    font-size:
        28px;
}

.plan-hero p {

    max-width:
        600px;

    margin:
        0;

    color:
        rgba(255,255,255,.65);

    font-size:
        12px;
}

.plan-target {

    text-align:
        right;
}

.plan-target span {

    display:
        block;

    color:
        rgba(255,255,255,.55);

    font-size:
        8px;

    font-weight:
        850;

    letter-spacing:
        1.3px;
}

.plan-target strong {

    font-size:
        35px;
}

.plan-target small {

    color:
        rgba(255,255,255,.60);
}

.target-grid {

    display:
        grid;

    grid-template-columns:
        repeat(4,1fr);

    gap:
        12px;

    margin-bottom:
        15px;
}

.target-card {

    padding:
        18px;

    border:
        1px solid #E6E9F0;

    border-radius:
        17px;

    background:
        white;
}

.target-card span {

    font-size:
        20px;
}

.target-card small {

    display:
        block;

    margin-top:
        10px;

    color:
        #98A0B1;

    font-size:
        8px;

    font-weight:
        850;

    letter-spacing:
        1px;
}

.target-card strong {

    color:
        #111827;

    font-size:
        24px;
}

.target-card em {

    color:
        #98A0B1;

    font-size:
        10px;

    font-style:
        normal;
}

.meal-grid {

    display:
        grid;

    grid-template-columns:
        repeat(2,1fr);

    gap:
        13px;
}

.meal-card {

    display:
        flex;

    gap:
        15px;

    padding:
        20px;

    border:
        1px solid #E6E9F0;

    border-radius:
        18px;

    background:
        white;
}

.meal-icon {

    width:
        50px;

    height:
        50px;

    min-width:
        50px;

    display:
        flex;

    justify-content:
        center;

    align-items:
        center;

    border-radius:
        14px;

    background:
        #F0EFFF;

    font-size:
        23px;
}

.meal-content h3 {

    margin:
        4px 0;

    color:
        #111827;

    font-size:
        18px;
}

.meal-content p {

    margin:
        0;

    color:
        #98A0B1;

    font-size:
        11px;
}

.meal-data {

    display:
        flex;

    gap:
        7px;

    flex-wrap:
        wrap;

    margin-top:
        12px;
}

.meal-data span {

    padding:
        5px 8px;

    border-radius:
        7px;

    color:
        #687286;

    background:
        #F6F7FA;

    font-size:
        9px;
}

.plan-note {

    display:
        flex;

    gap:
        13px;

    margin-top:
        15px;

    padding:
        17px;

    border-radius:
        14px;

    background:
        #F5F5FF;
}

.plan-note > span {

    color:
        #6C63FF;

    font-size:
        20px;
}

.plan-note strong {

    color:
        #374151;

    font-size:
        12px;
}

.plan-note p {

    margin:
        3px 0 0;

    color:
        #7B8497;

    font-size:
        10px;
}


/* ============================================================
   RESPONSIVE
   ============================================================ */

@media(max-width:1050px) {

    .metric-grid {

        grid-template-columns:
            repeat(2,1fr);
    }

    .analytics-grid {

        grid-template-columns:
            1fr;
    }

    .profile-strip {

        grid-template-columns:
            repeat(2,1fr);
    }

    .profile-strip > div:nth-child(2) {
        border-right:none;
    }

    .profile-strip > div:nth-child(-n+2) {
        border-bottom:
            1px solid #ECEEF2;
    }

    .meal-grid {

        grid-template-columns:
            1fr;
    }

}

@media(max-width:700px) {

    .gradio-container {
        padding:
            10px !important;
    }

    .app-header {
        padding:
            28px;
    }

    .app-header h1 {
        color: #FFFFFF !important;
        font-size:
            30px;
    }

    .dash-top {
        display:
            block;
    }

    .live-status {
        display:
            inline-flex;

        margin-top:
            14px;
    }

    .metric-grid {

        grid-template-columns:
            1fr;
    }

    .energy-body {

        flex-direction:
            column;

        align-items:
            flex-start;
    }

    .macro-layout {

        flex-direction:
            column;
    }

    .meal-timeline {

        overflow-x:
            auto;

        justify-content:
            flex-start;
    }

    .profile-strip {

        grid-template-columns:
            1fr;
    }

    .profile-strip > div {

        border-right:
            none !important;

        border-bottom:
            1px solid #ECEEF2;
    }

    .target-grid {

        grid-template-columns:
            repeat(2,1fr);
    }

    .plan-hero {

        display:
            block;
    }

    .plan-target {

        margin-top:
            20px;

        text-align:
            left;
    }

}

"""


# ================================================================
# 34. GRADIO APPLICATION
# ================================================================

with gr.Blocks(
    title="NutriAI — Intelligent Nutrition OS"
) as app:


    # ------------------------------------------------------------
    # HEADER
    # ------------------------------------------------------------

    gr.HTML("""

<div class="app-header">

    <h1 style="color:#FFFFFF !important;">
        ✦ NutriAI
    </h1>

    <p>
        Intelligent Nutrition OS
        &nbsp; • &nbsp;
        Personal Nutrition
        &nbsp; • &nbsp;
        Real Data Analytics
    </p>

</div>

""")


    # ============================================================
    # DASHBOARD
    # ============================================================

    with gr.Tab("◈  Overview"):

        refresh_dashboard = gr.Button(
            "↻  Refresh Live Dashboard",
            variant="primary"
        )

        dashboard_html = gr.HTML(
            value=build_dashboard_html()
        )

        # IMPORTANT: initialize the Plot components with real figures.
        # Without an initial value Gradio renders an empty chart area until
        # the refresh event is fired.
        initial_calorie_chart, initial_macro_chart = build_dashboard_charts()

        with gr.Row():

            calorie_chart = gr.Plot(
                value=initial_calorie_chart,
                label="7-Day Calorie Trend"
            )

            macro_chart = gr.Plot(
                value=initial_macro_chart,
                label="Macro Distribution"
            )

        gr.Markdown(
            "### Today's Recorded Intake"
        )

        dashboard_table = gr.Dataframe(
            value=refresh_today_table(),
            interactive=False,
            wrap=True
        )


        def refresh_dashboard_all():

            return (
                build_dashboard_html(),
                *build_dashboard_charts(),
                refresh_today_table()
            )


        refresh_dashboard.click(
            refresh_dashboard_all,
            inputs=[],
            outputs=[
                dashboard_html,
                calorie_chart,
                macro_chart,
                dashboard_table
            ]
        )


    # ============================================================
    # ACCOUNT
    # ============================================================

    with gr.Tab("⌁  Account"):

        gr.Markdown(
            """
## Account

Create a local NutriAI account or sign in to
access your personalized dashboard.
"""
        )

        with gr.Row():

            with gr.Column():

                gr.Markdown(
                    "### Create Account"
                )

                register_username = gr.Textbox(
                    label="Username"
                )

                register_password = gr.Textbox(
                    label="Password",
                    type="password"
                )

                register_button = gr.Button(
                    "Create Account",
                    variant="primary"
                )

                register_result = gr.Markdown()

                register_button.click(
                    register_user,
                    inputs=[
                        register_username,
                        register_password
                    ],
                    outputs=[
                        register_result
                    ]
                )


            with gr.Column():

                gr.Markdown(
                    "### Sign In"
                )

                login_username = gr.Textbox(
                    label="Username"
                )

                login_password = gr.Textbox(
                    label="Password",
                    type="password"
                )

                login_button = gr.Button(
                    "Sign In",
                    variant="primary"
                )

                logout_button = gr.Button(
                    "Sign Out"
                )

                login_result = gr.Markdown()

                login_button.click(
                    login_user,
                    inputs=[
                        login_username,
                        login_password
                    ],
                    outputs=[
                        login_result
                    ]
                ).then(
                    refresh_dashboard_all,
                    inputs=[],
                    outputs=[
                        dashboard_html,
                        calorie_chart,
                        macro_chart,
                        dashboard_table
                    ]
                )

                logout_button.click(
                    logout_user,
                    inputs=[],
                    outputs=[
                        login_result
                    ]
                ).then(
                    refresh_dashboard_all,
                    inputs=[],
                    outputs=[
                        dashboard_html,
                        calorie_chart,
                        macro_chart,
                        dashboard_table
                    ]
                )


    # ============================================================
    # PROFILE
    # ============================================================

    with gr.Tab("◎  Profile"):

        gr.Markdown(
            """
## Personal Nutrition Profile

Your profile drives the BMR, TDEE, calorie and
macronutrient calculations used by the dashboard.
"""
        )

        with gr.Row():

            profile_name = gr.Textbox(
                label="Full Name"
            )

            profile_age = gr.Number(
                label="Age",
                value=22
            )

            profile_sex = gr.Dropdown(
                choices=[
                    "Female",
                    "Male"
                ],
                value="Female",
                label="Sex"
            )

        with gr.Row():

            profile_height = gr.Number(
                label="Height (cm)",
                value=165
            )

            profile_weight = gr.Number(
                label="Weight (kg)",
                value=60
            )

            profile_activity = gr.Dropdown(
                choices=list(
                    ACTIVITY_FACTORS.keys()
                ),
                value="Moderately Active",
                label="Activity Level"
            )

        with gr.Row():

            profile_diet = gr.Dropdown(
                choices=[
                    "vegetarian",
                    "vegan",
                    "non-vegetarian"
                ],
                value="vegetarian",
                label="Diet Preference"
            )

            profile_goal = gr.Dropdown(
                choices=[
                    "Balanced Eating",
                    "Weight Loss",
                    "Weight Gain",
                    "Muscle & Fitness"
                ],
                value="Balanced Eating",
                label="Nutrition Goal"
            )

        profile_allergies = gr.Textbox(
            label="Allergies",
            placeholder="Example: nuts, lactose, gluten"
        )

        with gr.Row():

            profile_cuisine = gr.Dropdown(
                choices=[
                    "Indian",
                    "North Indian",
                    "South Indian",
                    "General"
                ],
                value="Indian",
                label="Preferred Cuisine"
            )

            profile_budget = gr.Number(
                label="Approx. Daily Food Budget",
                value=300
            )

        save_profile_button = gr.Button(
            "Save Nutrition Profile",
            variant="primary"
        )

        profile_result = gr.Markdown()

        save_profile_button.click(
            save_profile,
            inputs=[
                profile_name,
                profile_age,
                profile_sex,
                profile_height,
                profile_weight,
                profile_activity,
                profile_diet,
                profile_goal,
                profile_allergies,
                profile_cuisine,
                profile_budget
            ],
            outputs=[
                profile_result
            ]
        )


    # ============================================================
    # AI PLAN
    # ============================================================

    with gr.Tab("✦  Nutrition Plan"):

        gr.Markdown(
            """
## Personalized Nutrition Blueprint

Generate a meal plan from the profile currently
entered above.
"""
        )

        generate_button = gr.Button(
            "✦ Generate Personalized Plan",
            variant="primary"
        )

        generated_plan = gr.HTML()

        plan_state = gr.State(
            value=None
        )

        generate_button.click(
            generate_user_plan,
            inputs=[
                profile_name,
                profile_age,
                profile_sex,
                profile_height,
                profile_weight,
                profile_activity,
                profile_diet,
                profile_goal,
                profile_allergies,
                profile_cuisine,
                profile_budget
            ],
            outputs=[
                generated_plan,
                plan_state
            ]
        )

        gr.Markdown(
            "### Export"
        )

        export_button = gr.Button(
            "↓ Export Latest Plan as CSV"
        )

        export_output = gr.File()

        export_button.click(
            export_latest_plan,
            inputs=[],
            outputs=[
                export_output
            ]
        )


    # ============================================================
    # INTAKE TRACKER
    # ============================================================

    with gr.Tab("＋  Intake"):

        gr.Markdown(
            """
## Food Intake Tracker

Every entry is saved in the SQLite database and
is used by the dashboard analytics.
"""
        )

        with gr.Row():

            intake_food = gr.Dropdown(
                choices=FOOD_NAMES,
                value="Oats",
                label="Food"
            )

            intake_grams = gr.Number(
                value=100,
                label="Quantity (g)"
            )

            intake_date = gr.Textbox(
                value=str(date.today()),
                label="Date",
                placeholder="YYYY-MM-DD"
            )

        add_intake_button = gr.Button(
            "＋ Record Intake",
            variant="primary"
        )

        intake_result = gr.Markdown()

        add_intake_button.click(
            add_intake,
            inputs=[
                intake_food,
                intake_grams,
                intake_date
            ],
            outputs=[
                intake_result
            ]
        )

        gr.Markdown(
            "### Recorded Intake"
        )

        refresh_intake_button = gr.Button(
            "↻ Refresh Intake Table"
        )

        intake_table = gr.Dataframe(
            value=refresh_today_table(),
            interactive=False,
            wrap=True
        )

        refresh_intake_button.click(
            refresh_today_table,
            inputs=[],
            outputs=[
                intake_table
            ]
        )

        gr.Markdown(
            "### Complete History"
        )

        history_button = gr.Button(
            "View Complete Intake History"
        )

        history_table = gr.Dataframe(
            interactive=False,
            wrap=True
        )

        history_button.click(
            get_all_intake,
            inputs=[],
            outputs=[
                history_table
            ]
        )


    # ============================================================
    # SAVED PLANS
    # ============================================================

    with gr.Tab("▣  Saved Plans"):

        gr.Markdown(
            """
## Saved Nutrition Plans

Your generated plans are stored locally in SQLite.
"""
        )

        plans_button = gr.Button(
            "↻ Load Saved Plans"
        )

        plans_table = gr.Dataframe(
            interactive=False,
            wrap=True
        )

        plans_button.click(
            get_saved_plans,
            inputs=[],
            outputs=[
                plans_table
            ]
        )






# ================================================================

# 35. LAUNCH

# ================================================================

#

# IMPORTANT GRADIO 6 FIX:

#

# theme and css belong in launch().

#

# They are NOT passed to gr.Blocks().

#

# ================================================================

print()
print("=" * 70)
print("✦ NUTRIAI — APPLICATION READY")
print("=" * 70)
print()
print("✓ Colab-safe Pandas")
print("✓ Colab-safe NumPy")
print("✓ Gradio 6")
print("✓ SQLite persistence")
print("✓ Real database-driven dashboard")
print("✓ Actual 7-day analytics")
print("✓ Macro visualization")
print("✓ Personalized meal planner")
print("✓ Intake tracker")
print("✓ CSV export")
print("✓ Responsive professional UI")
print()
print("Launching application...")
print("=" * 70)

app.launch(
share=True,
debug=False,
css=CSS,
theme=gr.themes.Soft(
primary_hue="indigo",
secondary_hue="violet",
neutral_hue="slate"
)
)



NUTRIAI — ENVIRONMENT
Python : 3.13.15
Pandas : 2.2.3
NumPy  : 2.1.3
Gradio : 6.27.0

✦ NUTRIAI — APPLICATION READY

✓ Colab-safe Pandas
✓ Colab-safe NumPy
✓ Gradio 6
✓ SQLite persistence
✓ Real database-driven dashboard
✓ Actual 7-day analytics
✓ Macro visualization
✓ Personalized meal planner
✓ Intake tracker
✓ CSV export
✓ Responsive professional UI

Launching application...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0ed1dcf8d1b07d226d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
